# SAR → EO — Phase 1 Training
**ResNet50-UNet + CBAM  ·  Multi-Scale PatchGAN  ·  L1 + Adv + FFT + VGG + MS-SSIM**

---

### Before running

1. **Settings → Accelerator → GPU T4 x2** (or P100)
2. **Settings → Internet → On** — needed to clone the repo and fetch ImageNet weights
3. **Add Input** → search `sentinel12-image-pairs-segregated-by-terrain` → Add
4. **Session 2 only:** also Add Input → *Your Work* → this notebook's previous output

### The 12-hour rule

Kaggle kills the session at 12h. This notebook trains **75 epochs per session**:

| Session | Epochs | What to do at the end |
|---------|--------|----------------------|
| 1 | 1 → 75 | Click **Save Version** *before* the 12h mark |
| 2 | 76 → 150 | Add Session 1's output as input, rerun top to bottom |

Checkpoints save every 5 epochs, so a timeout costs at most 5 epochs.

### Cell 5 is a hard gate

It verifies no source scene appears in more than one split. SEN1-2 tiles each
scene on a stride grid, so neighbouring patches overlap on the ground — a
per-patch split leaks and inflates every metric. **If it fails, training refuses
to start.** That is intentional.

## 1 · Repo and dependencies

In [ ]:
import subprocess, os, sys, yaml, shutil, glob, re, torch

REPO = "/kaggle/working/sar2eo"
if os.path.exists(REPO):
    subprocess.run(f"cd {REPO} && git pull --quiet", shell=True)
    print("Repo updated")
else:
    subprocess.run(
        f"git clone --quiet https://github.com/Trafalgar-2006/sar2eo.git {REPO}",
        shell=True, check=True)
    print("Repo cloned")

sys.path.insert(0, REPO)
os.chdir(REPO)

subprocess.run("pip install -q lpips pytorch-fid", shell=True, check=True)

commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"]).decode().strip()
print(f"Deps installed | commit {commit}")

## 2 · GPU and dataset discovery

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Settings -> Accelerator -> GPU T4 x2")

print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

INPUT_ROOT   = "/kaggle/input"
TERRAIN_KEYS = {"agri", "urban", "grassland", "barrenland",
                "forest", "water", "mountain"}
KAGGLE_DATA  = None

# The dataset root is whichever directory holds >=2 terrain subfolders.
for dirpath, dirnames, _ in os.walk(INPUT_ROOT):
    found = {d.lower() for d in dirnames} & TERRAIN_KEYS
    if len(found) >= 2:
        KAGGLE_DATA = dirpath
        print(f"\nDataset : {KAGGLE_DATA}")
        print(f"Terrains: {sorted(found)}")
        break

if KAGGLE_DATA is None:
    print("\nCould not find the dataset. /kaggle/input contains:")
    for dirpath, dirnames, _ in os.walk(INPUT_ROOT):
        depth = dirpath.replace(INPUT_ROOT, "").count(os.sep)
        if depth <= 3:
            print("  " * depth + os.path.basename(dirpath) + "/")
    raise FileNotFoundError(
        "Add Input -> search 'sentinel12-image-pairs-segregated-by-terrain'")

## 3 · Auto-resume from a previous session

Scans `/kaggle/input` for `.pth` files. Finds nothing on Session 1 and starts
from scratch; on Session 2 it copies Session 1's checkpoints into place so
`train()` picks up where it left off — including optimiser, LR schedule and
best-so-far validation loss.

In [ ]:
CKPT_DST = "/kaggle/working/checkpoints/full"
os.makedirs(CKPT_DST, exist_ok=True)

restored = 0
for dirpath, _, filenames in os.walk(INPUT_ROOT):
    pth_files = [f for f in filenames if f.endswith(".pth")]
    if not pth_files:
        continue
    print(f"Found previous checkpoints in: {dirpath}")
    for f in pth_files:
        dst = os.path.join(CKPT_DST, f)
        if not os.path.exists(dst):
            shutil.copy2(os.path.join(dirpath, f), dst)
            restored += 1
            print(f"  restored {f}")
    break

# Parse "epoch_075.pth" -> 75. Plain string ops, no regex escaping to get wrong.
done = []
for f in glob.glob(f"{CKPT_DST}/epoch_*.pth"):
    stem = os.path.basename(f).replace("epoch_", "").replace(".pth", "")
    if stem.isdigit():
        done.append(int(stem))
LAST_EPOCH = max(done) if done else 0

if LAST_EPOCH:
    print(f"\nRESUMING - last completed epoch was {LAST_EPOCH}")
else:
    print("\nFRESH START - no previous checkpoints found")

## 4 · Configuration

In [ ]:
# The LR schedule spans the FULL run, so TOTAL_EPOCHS never changes between
# sessions — otherwise the cosine would rescale and the LR would jump back up
# on resume. SESSION_EPOCHS just caps how much this 12h session executes.
TOTAL_EPOCHS   = 150
SESSION_EPOCHS = 75

config = {
    "model": {
        "input_channels":  1,       # VV only. Set 2 for VV+VH dual-pol.
        "output_channels": 3,
        "base_ch":         64,
        "use_attention":   True,    # CBAM on every skip connection
        "pretrained_encoder": True,
        "gradient_checkpointing": False,   # flip to True if OOM
        "n_scales_D":      3,
        "n_layers_D":      3,
    },
    "training": {
        "epochs":              TOTAL_EPOCHS,
        "session_epoch_limit": SESSION_EPOCHS,
        "batch_size":   8,
        "lr_encoder":   2e-5,       # pretrained ResNet50 - gentle
        "lr_decoder":   2e-4,       # random init - normal
        "lr_discriminator": 2e-4,
        "beta1": 0.5, "beta2": 0.999,
        "warmup_epochs": 5,
        "lr_min":        1e-6,
        "gradient_clip_norm": 1.0,
        "ema_decay":     0.999,
        "mixed_precision": True,
        "save_freq":     5,         # a timeout costs at most 5 epochs
        "val_freq":      10,
        "seed":          42,
    },
    "loss": {
        "lambda_l1": 100.0, "lambda_adv": 1.0, "lambda_fft": 10.0,
        "lambda_vgg": 10.0, "lambda_ssim": 5.0,
    },
    "active_ablation": "full",
    "data": {
        "dataset_type": "kaggle",
        # Group patches by the scene they were tiled from. SEN1-2 cuts each
        # scene on a stride grid, so neighbouring patches overlap on the ground.
        # A per-patch ("random") split puts near-duplicates in train AND test.
        "split_strategy": "scene",
        "sen12_root":    "./data/SEN1-2",
        "train_seasons": ["spring", "summer", "fall"],
        "val_seasons":   ["winter"],
        "test_seasons":  ["winter"],
        "kaggle_root":   KAGGLE_DATA,
        "train_terrain": ["agri", "barrenland", "grassland"],
        "val_terrain":   ["urban"],
        "test_terrain":  ["urban"],
        "image_size":    256,
        "subset_size":   None,
        "num_workers":   2,
    },
    "augmentation": {
        "horizontal_flip": True, "vertical_flip": True, "rotation_90": True,
        "sar_gaussian_noise": True, "eo_brightness_jitter": True,
    },
    "paths": {
        "checkpoint_dir": "/kaggle/working/checkpoints",
        "output_dir":     "/kaggle/working/outputs",
        "log_dir":        "/kaggle/working/logs",
    },
}

CFG_PATH = f"{REPO}/config_kaggle.yaml"
with open(CFG_PATH, "w") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

_this_session = min(SESSION_EPOCHS, TOTAL_EPOCHS - LAST_EPOCH)
print(f"Config written -> {CFG_PATH}")
print(f"  total run     : {TOTAL_EPOCHS} epochs")
print(f"  done already  : {LAST_EPOCH}")
print(f"  this session  : {_this_session} epochs ({LAST_EPOCH + 1} -> {LAST_EPOCH + _this_session})")
print(f"  split         : {config['data']['split_strategy']}")
print(f"  est. runtime  : {_this_session * 9.5 / 60:.1f} h on T4")

## 5 · Leakage audit — hard gate

No source scene may appear in more than one split. If any does, the test set
contains ground the model trained on and every metric below is inflated.
This cell raises rather than letting you waste a GPU session.

In [ ]:
from data.dataloader import SARtoEODataset, _scene_key

audit_cfg = yaml.safe_load(open(CFG_PATH))
scenes = {}

for split in ("train", "val", "test"):
    ds = SARtoEODataset(audit_cfg, split=split, augment=False)
    scenes[split] = {_scene_key(p[0]) for p in ds.pairs}
    print(f"  {split:<6}: {len(ds.pairs):>7,} patches from {len(scenes[split]):>5,} scenes")

print()
leaked = False
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    shared = scenes[a] & scenes[b]
    print(f"  {a:<6} vs {b:<5}: "
          f"{'OK' if not shared else f'LEAK - {len(shared)} shared scenes'}")
    leaked |= bool(shared)

if leaked:
    raise RuntimeError(
        "Split is leaking - test scenes also appear in train. Refusing to "
        "train: the resulting metrics would be meaningless. Check that "
        "split_strategy is 'scene' in cell 4.")

print("\nPASSED - splits are scene-disjoint")

## 6 · Smoke test

One forward pass through G and D to catch shape or VRAM problems before the long run.

In [ ]:
from models.generator import UNetGenerator
from models.discriminator import MultiScaleDiscriminator

device = torch.device("cuda")
G = UNetGenerator(in_channels=1, out_channels=3,
                  use_attention=True, pretrained=True).to(device)
D = MultiScaleDiscriminator().to(device)

with torch.no_grad(), torch.amp.autocast(device_type="cuda"):
    out  = G(torch.randn(2, 1, 256, 256, device=device))
    disc = D(torch.randn(2, 1, 256, 256, device=device),
             torch.randn(2, 3, 256, 256, device=device))

print(f"G out    : {tuple(out.shape)}  range [{out.min():.2f}, {out.max():.2f}]")
print(f"D scales : {[tuple(d.shape) for d in disc]}")
print(f"G params : {sum(p.numel() for p in G.parameters()):,}")
print(f"D params : {sum(p.numel() for p in D.parameters()):,}")
print(f"VRAM     : {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

del G, D, out, disc
torch.cuda.empty_cache()

## 7 · Train

Long-running. Checkpoints land in `/kaggle/working/checkpoints/full/` every 5
epochs, and `outputs/losses_full.csv` is rewritten every epoch — so an
interrupted session still leaves usable artefacts.

**Watch the clock.** If this is Session 1, hit **Save Version** before 12h.

In [ ]:
from train import train, load_config, make_dirs

cfg = load_config(CFG_PATH)
make_dirs(cfg)

print("=" * 62)
print(f" Phase 1 - this session runs {_this_session} epoch(s), target {TOTAL_EPOCHS}")
print("=" * 62)

G = train(cfg)

print("\nTraining complete for this session.")
if LAST_EPOCH + _this_session < TOTAL_EPOCHS:
    print("\n" + "=" * 62)
    print(" SESSION DONE - CLICK 'SAVE VERSION' NOW")
    print("=" * 62)
    print(" Next session: Add Input -> Your Work -> this notebook's output,")
    print(f" then rerun. It resumes at epoch {LAST_EPOCH + _this_session + 1}.")
else:
    print(f"\nAll {TOTAL_EPOCHS} epochs done. Run the eval cell below.")

## 8 · Evaluate on the held-out test split

In [ ]:
from eval import run_inference_to_dir, evaluate_dirs

WEIGHTS  = "/kaggle/working/checkpoints/full/best.pth"
PRED_DIR = "/kaggle/working/outputs/eval_preds_test"
GT_DIR   = "/kaggle/working/outputs/eval_gt_test"
OUT_CSV  = "/kaggle/working/outputs/metrics_test.csv"

run_inference_to_dir(CFG_PATH, WEIGHTS, "test", PRED_DIR, GT_DIR, use_tta=False)
metrics = evaluate_dirs(PRED_DIR, GT_DIR, OUT_CSV, split="test")

print("\n" + "=" * 46)
print("  FINAL METRICS  (scene-disjoint test split)")
print("=" * 46)
print(f"  SSIM  (higher better) : {metrics['ssim']:.4f}")
print(f"  PSNR  (higher better) : {metrics['psnr']:.2f} dB")
print(f"  LPIPS (lower  better) : {metrics['lpips']:.4f}")
print(f"  FID   (lower  better) : {metrics['fid']:.2f}")
print("=" * 46)
print("\nThese are honest numbers - no train/test scene overlap.")

## 9 · Download

From the **Output** tab, grab:

| File | Why |
|------|-----|
| `checkpoints/full/best.pth` | the trained model — put it in `checkpoints/full/` locally |
| `outputs/metrics_test.csv` | the numbers for the README results table |
| `outputs/losses_full.csv` | loss history, for `plot_results.py` |
| `outputs/loss_curve_full.png` | training curves |
| `outputs/samples/full/` | qualitative SAR / generated / ground-truth triplets |
| `logs/full_steps.jsonl` | per-step losses |

Then locally:

```bash
python eval.py --config config.yaml --weights checkpoints/full/best.pth --split test
python plot_results.py
```

**Next:** Phase 2 (conditional diffusion) via `kaggle_train_diffusion.py` — only
worth starting once these Phase 1 numbers look solid.